In [1]:
"""
=============================================================================
FLOOD STREAMFLOW PREDICTION — v9
=============================================================================

CORE INSIGHT FROM v8 DIAGNOSIS
────────────────────────────────
v8 val KGE = 0.99726. Remaining error is almost entirely from flood peaks
(error fan at Q > 1000 cumecs in error_analysis.png).

ROOT CAUSE: predicting log1p(Q_tomorrow) directly means the model spends
most of its capacity learning "tomorrow ≈ today" (the trivial autocorrelation).
The flood events (the hard 10% that determine your KGE) get insufficient
model attention because their contribution to RMSE loss is diluted.

FIX 1: DELTA TARGET  ← biggest single change
────────────────────────────────────────────────
Instead of predicting:  log1p(Q_tomorrow)
Predict:               log1p(Q_tomorrow) - log1p(Q_today)  = log_delta

This is the LOG-CHANGE in flow. Properties:
  • ~0 for 90% of rows (stable flow) → easy, model ignores these
  • Large positive for flood onset → model focuses here
  • Large negative for recession → model focuses here
  • Target variance drops ~10x → model has to learn actual dynamics
  • Reconstruction: Q_pred = expm1(log1p(Q_today) + pred_delta)

The r component of KGE (correlation) improves because predicting
rank-order of CHANGES is much easier than rank-ordering absolute values.

FIX 2: PER-STATION MODELS FOR WORST 10 STATIONS
──────────────────────────────────────────────────
Stations 328, 242, 1, 142, 197, 290, 252, 219, 309, 213 identified
from per_station_kge.csv with KGE < 0.991.
Each gets its own dedicated LGB model trained only on that station's data
with higher capacity (more leaves, more trees).

FIX 3: FLOOD-WEIGHTED LOSS  (LightGBM custom objective)
──────────────────────────────────────────────────────────
Weights high-flow rows more heavily so model doesn't under-fit flood peaks.
weight = 1 + 4 * (Q_today / station_mean_flow).clip(0, 5)
→ flood rows get up to 5x more gradient signal than baseflow rows.

EVERYTHING ELSE: unchanged from v8 (don't break what works)

EXPECTED GAIN: +0.001 to +0.003 KGE from each fix independently.
=============================================================================
"""

import gc
import sys
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from scipy.optimize import minimize
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings("ignore")


def flush(msg=""):
    print(msg, flush=True)
    sys.stdout.flush()


# ─────────────────────────────────────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────────────────────────────────────
TRAIN_PATH = r"E:\Kaggle\steamflow prediction\train_flood.csv"
TEST_PATH  = r"E:\Kaggle\steamflow prediction\test_flood.csv"
SUB_DIR    = r"E:\Kaggle\steamflow prediction\v9"
SUB_PATH   = os.path.join(SUB_DIR, "submission.csv")
os.makedirs(SUB_DIR, exist_ok=True)
DIAG_DIR   = os.path.join(SUB_DIR, "diagnostics")
os.makedirs(DIAG_DIR, exist_ok=True)

_TARGET = "streamflow_tomorrow_cumecs"

# From per_station_kge.csv — stations with KGE < 0.991
WORST_STATIONS = [328, 242, 1, 142, 197, 290, 252, 219, 309, 213]


# ─────────────────────────────────────────────────────────────────────────────
# 1. METRICS
# ─────────────────────────────────────────────────────────────────────────────
def kge(y_obs: np.ndarray, y_sim: np.ndarray) -> float:
    r     = np.corrcoef(y_obs, y_sim)[0, 1]
    alpha = y_sim.std()  / (y_obs.std()  + 1e-10)
    beta  = y_sim.mean() / (y_obs.mean() + 1e-10)
    return float(1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2))


def kge_components(y_obs: np.ndarray, y_sim: np.ndarray) -> dict:
    r     = np.corrcoef(y_obs, y_sim)[0, 1]
    alpha = y_sim.std()  / (y_obs.std()  + 1e-10)
    beta  = y_sim.mean() / (y_obs.mean() + 1e-10)
    score = 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)
    return dict(KGE=round(score, 6), r=round(r, 4),
                alpha=round(alpha, 4), beta=round(beta, 4))


def nse(y_obs: np.ndarray, y_sim: np.ndarray) -> float:
    return float(1 - np.sum((y_obs - y_sim)**2) /
                 np.sum((y_obs - y_obs.mean())**2))


# ─────────────────────────────────────────────────────────────────────────────
# 2. DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────
_F32 = [
    "streamflow_today_cumecs", "streamflow_anomaly_zscore",
    "flow_rate_of_change", "flow_velocity_km_per_day",
    "antecedent_rain_3d_sum", "antecedent_rain_7d_sum",
    "antecedent_rain_15d_sum", "antecedent_rain_30d_sum",
    "antecedent_rain_60d", "antecedent_rain_ewm",
    "rainfall_anomaly_zscore",
    "upstream_rain_mean_scaled", "upstream_rain_weighted_scaled",
    "upstream_rain_lagged_dist_sink",
    "soil_saturation_score", "antecedent_saturation_interaction",
    "monsoon_cumulative_rain", "monsoon_intensity",
    "dist_to_outlet_scaled", "upstream_area_scaled",
    "slope_scaled", "slope_uav_scaled",
    "forest_cover_scaled", "urban_cover_scaled",
    "rain_soilmoisture_interaction", "rain_urban_interaction",
    "rain_slope_interaction", "rain_basinsize_interaction",
    "uparea_rain_interaction",
]


def load_data(path: str, is_train: bool = True) -> pd.DataFrame:
    flush(f"\nLoading {'train' if is_train else 'test'}...")
    dtype = {c: "float32" for c in _F32}
    dtype.update({"row_id": "int32", "month": "int8",
                  "day_of_year": "int16",
                  "is_post_monsoon_saturated": "int8"})
    df = pd.read_csv(path, dtype=dtype)
    if is_train:
        df[_TARGET] = df[_TARGET].astype("float32")
    mb = df.memory_usage(deep=True).sum() / 1e6
    flush(f"  {df.shape[0]:,} rows × {df.shape[1]} cols | {mb:.0f} MB")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# 3. STATION IDENTIFICATION
# ─────────────────────────────────────────────────────────────────────────────
_STATIC = ["dist_to_outlet_scaled", "upstream_area_scaled",
           "slope_scaled", "slope_uav_scaled",
           "forest_cover_scaled", "urban_cover_scaled"]


def add_station_id(df: pd.DataFrame, enc=None) -> tuple:
    combo = df[_STATIC].round(4).astype(str).agg("|".join, axis=1)
    if enc is None:
        enc = LabelEncoder()
        df["station_id"] = enc.fit_transform(combo).astype("int16")
    else:
        known = set(enc.classes_)
        combo = combo.where(combo.isin(known), other=enc.classes_[0])
        df["station_id"] = enc.transform(combo).astype("int16")
    return df, enc


# ─────────────────────────────────────────────────────────────────────────────
# 4. STATION TARGET STATS
# ─────────────────────────────────────────────────────────────────────────────
def add_station_stats(train_df: pd.DataFrame,
                      test_df: pd.DataFrame) -> tuple:
    stats = (train_df.groupby("station_id")[_TARGET]
             .agg(station_mean_flow="mean",
                  station_std_flow="std",
                  station_p90_flow=lambda x: np.percentile(x, 90))
             .reset_index())
    stats["station_std_flow"] = stats["station_std_flow"].fillna(0)

    gm  = float(train_df[_TARGET].mean())
    gs  = float(train_df[_TARGET].std())
    gp90 = float(np.percentile(train_df[_TARGET], 90))

    for df in (train_df, test_df):
        df.drop(columns=[c for c in
                         ["station_mean_flow", "station_std_flow",
                          "station_p90_flow"]
                         if c in df.columns], inplace=True)
    train_df = train_df.merge(stats, on="station_id", how="left")
    test_df  = test_df.merge(stats,  on="station_id", how="left")
    for df in (train_df, test_df):
        df["station_mean_flow"] = df["station_mean_flow"].fillna(gm).astype("float32")
        df["station_std_flow"]  = df["station_std_flow"].fillna(gs).astype("float32")
        df["station_p90_flow"]  = df["station_p90_flow"].fillna(gp90).astype("float32")

    flush(f"  station_mean_flow range: "
          f"[{train_df['station_mean_flow'].min():.0f}, "
          f"{train_df['station_mean_flow'].max():.0f}]")
    return train_df, test_df


# ─────────────────────────────────────────────────────────────────────────────
# 5. LAG FEATURES (same as v8 — already working)
# ─────────────────────────────────────────────────────────────────────────────
def add_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    flush("  Adding lag features...")
    df = df.sort_values(["station_id", "row_id"]).reset_index(drop=True)
    grp_flow = df.groupby("station_id")["streamflow_today_cumecs"]
    grp_rain = df.groupby("station_id")["antecedent_rain_3d_sum"]

    for lag in range(1, 8):
        df[f"flow_lag_{lag}"] = (
            grp_flow.shift(lag)
            .fillna(df["streamflow_today_cumecs"])
            .astype("float32"))

    for lag in range(1, 4):
        df[f"rain_lag_{lag}"] = (
            grp_rain.shift(lag)
            .fillna(df["antecedent_rain_3d_sum"])
            .astype("float32"))

    df["flow_roll_mean_3d"] = (
        grp_flow.transform(lambda x: x.rolling(3,  min_periods=1).mean())
        .astype("float32"))
    df["flow_roll_mean_7d"] = (
        grp_flow.transform(lambda x: x.rolling(7,  min_periods=1).mean())
        .astype("float32"))
    df["flow_roll_max_7d"]  = (
        grp_flow.transform(lambda x: x.rolling(7,  min_periods=1).max())
        .astype("float32"))
    df["flow_roll_std_7d"]  = (
        grp_flow.transform(lambda x: x.rolling(7,  min_periods=2).std())
        .fillna(0).astype("float32"))
    df["rain_roll_std_7d"]  = (
        grp_rain.transform(lambda x: x.rolling(7,  min_periods=2).std())
        .fillna(0).astype("float32"))

    flush(f"  Shape after lags: {df.shape}")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# 6. FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────────────────
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    Q   = df["streamflow_today_cumecs"].astype("float64")
    doy = df["day_of_year"].astype("float64")
    mon = df["month"].astype("float64")
    sat = df["soil_saturation_score"].astype("float64")
    ups = df["upstream_rain_mean_scaled"].astype("float64")

    # ── Flow
    df["log_flow"]          = np.log1p(Q).astype("float32")
    df["log_flow_sq"]       = (np.log1p(Q) ** 2).astype("float32")
    df["rel_rate"]          = (df["flow_rate_of_change"] / (Q + 1)
                               ).clip(-10, 10).astype("float32")
    df["flow_accel_sign"]   = np.sign(df["rel_rate"]).astype("int8")
    df["flow_norm_station"] = (
        (Q - df["station_mean_flow"].astype("float64")) /
        (df["station_std_flow"].astype("float64") + 1)
    ).clip(-5, 10).astype("float32")

    # Log lags
    for lag in range(1, 8):
        col = f"flow_lag_{lag}"
        if col in df.columns:
            df[f"log_flow_lag_{lag}"] = np.log1p(
                df[col].astype("float64")).astype("float32")

    # Flow derivatives
    if "flow_lag_1" in df.columns:
        lag1 = df["flow_lag_1"].astype("float64")
        df["flow_d1"] = (Q - lag1).astype("float32")
        df["flow_log_d1"] = (np.log1p(Q) - np.log1p(lag1)).astype("float32")

    if "flow_lag_1" in df.columns and "flow_lag_2" in df.columns:
        d1 = Q - df["flow_lag_1"].astype("float64")
        d1_prev = (df["flow_lag_1"].astype("float64") -
                   df["flow_lag_2"].astype("float64"))
        df["flow_d2"] = (d1 - d1_prev).clip(-5000, 5000).astype("float32")

    # Flood regime flags
    if "flow_lag_1" in df.columns:
        lag1 = df["flow_lag_1"].astype("float64")
        df["is_rising"]      = (Q > lag1 * 1.05).astype("int8")
        df["is_flooding"]    = (Q > df["station_p90_flow"].astype("float64")).astype("int8")
        df["is_receding"]    = (Q < lag1 * 0.95).astype("int8")
        df["flow_lag1_ratio"] = (Q / (lag1 + 1)).clip(0, 10).astype("float32")

    if "flow_roll_mean_7d" in df.columns:
        df["flow_vs_roll7"]    = (Q - df["flow_roll_mean_7d"].astype("float64")).astype("float32")
        df["flow_spike_ratio"] = (Q / (df["flow_roll_mean_7d"].astype("float64") + 1)).clip(0, 20).astype("float32")

    if "flow_roll_max_7d" in df.columns:
        df["flow_pct_of_7d_max"] = (Q / (df["flow_roll_max_7d"].astype("float64") + 1)).clip(0, 1).astype("float32")

    # Time
    df["doy_sin"]      = np.sin(2 * np.pi * doy / 365).astype("float32")
    df["doy_cos"]      = np.cos(2 * np.pi * doy / 365).astype("float32")
    df["month_sin"]    = np.sin(2 * np.pi * mon / 12).astype("float32")
    df["month_cos"]    = np.cos(2 * np.pi * mon / 12).astype("float32")
    df["week_of_year"] = (doy // 7).clip(0, 52).astype("int8")

    # Rain
    r3  = df["antecedent_rain_3d_sum"].astype("float64")  + 1e-6
    r7  = df["antecedent_rain_7d_sum"].astype("float64")  + 1e-6
    r15 = df["antecedent_rain_15d_sum"].astype("float64") + 1e-6
    r30 = df["antecedent_rain_30d_sum"].astype("float64") + 1e-6
    r60 = df["antecedent_rain_60d"].astype("float64")     + 1e-6

    df["rain_ratio_3_30"]  = (r3  / r30).clip(0, 20).astype("float32")
    df["rain_ratio_7_30"]  = (r7  / r30).clip(0, 20).astype("float32")
    df["rain_ratio_3_60"]  = (r3  / r60).clip(0, 20).astype("float32")
    df["rain_ratio_15_60"] = (r15 / r60).clip(0, 20).astype("float32")
    df["rain_accel"]       = ((r3 / 3) - (r7 / 7)).astype("float32")
    df["log_rain_3d"]  = np.log1p(df["antecedent_rain_3d_sum"].astype("float64")).astype("float32")
    df["log_rain_7d"]  = np.log1p(df["antecedent_rain_7d_sum"].astype("float64")).astype("float32")
    df["log_rain_30d"] = np.log1p(df["antecedent_rain_30d_sum"].astype("float64")).astype("float32")
    df["log_rain_ewm"] = np.log1p(df["antecedent_rain_ewm"].astype("float64")).astype("float32")

    # Interactions
    df["ups_x_logflow"]   = (ups * np.log1p(Q)).astype("float32")
    df["ups_x_sat"]       = (ups * sat).astype("float32")
    df["logflow_x_sat"]   = (np.log1p(Q) * sat).astype("float32")
    df["preflood_alarm"]  = (
        df["streamflow_anomaly_zscore"].astype("float64") *
        df["rel_rate"].astype("float64") * sat
    ).clip(-50, 50).astype("float32")
    df["rain_anom_x_sat"] = (df["rainfall_anomaly_zscore"].astype("float64") * sat).astype("float32")

    # Monsoon
    df["in_peak_monsoon"] = ((doy >= 150) & (doy <= 270)).astype("int8")
    df["in_pre_monsoon"]  = ((doy >= 90)  & (doy < 150)).astype("int8")
    df["rain_load_ratio"] = (
        df["monsoon_cumulative_rain"].astype("float64") /
        (df["station_mean_flow"].astype("float64") + 1)
    ).clip(0, 100).astype("float32")

    df["is_worst_station"] = df["station_id"].isin(WORST_STATIONS).astype("int8")

    return df


# ─────────────────────────────────────────────────────────────────────────────
# 7. FEATURE SETS
# ─────────────────────────────────────────────────────────────────────────────
FEATURE_COLS = [
    "streamflow_today_cumecs", "streamflow_anomaly_zscore",
    "flow_rate_of_change", "flow_velocity_km_per_day",
    "antecedent_rain_3d_sum", "antecedent_rain_7d_sum",
    "antecedent_rain_15d_sum", "antecedent_rain_30d_sum",
    "antecedent_rain_60d", "antecedent_rain_ewm", "rainfall_anomaly_zscore",
    "upstream_rain_mean_scaled", "upstream_rain_weighted_scaled",
    "upstream_rain_lagged_dist_sink",
    "soil_saturation_score", "antecedent_saturation_interaction",
    "is_post_monsoon_saturated",
    "monsoon_intensity", "monsoon_cumulative_rain",
    "dist_to_outlet_scaled", "upstream_area_scaled",
    "slope_scaled", "slope_uav_scaled", "forest_cover_scaled", "urban_cover_scaled",
    "rain_soilmoisture_interaction", "rain_urban_interaction",
    "rain_slope_interaction", "rain_basinsize_interaction", "uparea_rain_interaction",
    "station_id", "station_mean_flow", "station_std_flow", "station_p90_flow",
    "log_flow", "log_flow_sq", "rel_rate", "flow_accel_sign", "flow_norm_station",
    "flow_lag_1", "flow_lag_2", "flow_lag_3", "flow_lag_4",
    "flow_lag_5", "flow_lag_6", "flow_lag_7",
    "rain_lag_1", "rain_lag_2", "rain_lag_3",
    "log_flow_lag_1", "log_flow_lag_2", "log_flow_lag_3",
    "log_flow_lag_4", "log_flow_lag_5", "log_flow_lag_6", "log_flow_lag_7",
    "flow_roll_mean_3d", "flow_roll_mean_7d", "flow_roll_max_7d",
    "flow_roll_std_7d", "rain_roll_std_7d",
    "flow_d1", "flow_d2", "flow_log_d1",
    "is_rising", "is_flooding", "is_receding",
    "flow_lag1_ratio", "flow_vs_roll7", "flow_spike_ratio", "flow_pct_of_7d_max",
    "is_worst_station",
    "week_of_year", "doy_sin", "doy_cos", "month_sin", "month_cos",
    "rain_ratio_3_30", "rain_ratio_7_30", "rain_ratio_3_60", "rain_ratio_15_60",
    "rain_accel", "log_rain_3d", "log_rain_7d", "log_rain_30d", "log_rain_ewm",
    "ups_x_logflow", "ups_x_sat", "logflow_x_sat",
    "preflood_alarm", "rain_anom_x_sat",
    "in_peak_monsoon", "in_pre_monsoon", "rain_load_ratio",
]

LGB_CAT_COLS = [
    "is_post_monsoon_saturated", "flow_accel_sign",
    "in_peak_monsoon", "in_pre_monsoon",
    "is_rising", "is_flooding", "is_receding", "is_worst_station",
]


def get_feature_cols(df):
    return [c for c in FEATURE_COLS if c in df.columns]


# ─────────────────────────────────────────────────────────────────────────────
# 8. DELTA TARGET HELPERS
#
#   target_delta = log1p(Q_tomorrow) - log1p(Q_today)
#   Reconstruction: Q_pred = expm1(log1p(Q_today) + pred_delta)
#
#   Why this works for KGE:
#   The r component of KGE measures rank correlation. Predicting
#   log-changes in flow is a much better-defined task than predicting
#   absolute log-flow, because the changes are driven by rain+soil physics
#   whereas the absolute level is driven mostly by autocorrelation.
# ─────────────────────────────────────────────────────────────────────────────
def make_delta_target(df: pd.DataFrame) -> np.ndarray:
    log_tomorrow = np.log1p(df[_TARGET].astype("float64"))
    log_today    = np.log1p(df["streamflow_today_cumecs"].astype("float64"))
    return (log_tomorrow - log_today).astype("float32").values


def reconstruct_from_delta(delta_pred: np.ndarray,
                           q_today: np.ndarray) -> np.ndarray:
    """Convert predicted log-delta back to Q_tomorrow."""
    log_today = np.log1p(q_today.astype("float64"))
    return np.expm1(log_today + delta_pred.astype("float64")).clip(0)


# ─────────────────────────────────────────────────────────────────────────────
# 9. FLOOD-WEIGHTED SAMPLE WEIGHTS
#
#   High-flow rows get up to 5x weight so the model doesn't under-fit peaks.
#   weight = 1 + 4 * clip(Q / station_mean, 0, 5)
#     baseflow (Q = mean): weight = 2
#     flood    (Q = 5×mean): weight = 5
#     extreme  (Q >> 5×mean): capped at weight = 5
# ─────────────────────────────────────────────────────────────────────────────
def make_sample_weights(df: pd.DataFrame) -> np.ndarray:
    ratio = (df["streamflow_today_cumecs"].astype("float64") /
             (df["station_mean_flow"].astype("float64") + 1)).clip(0, 5)
    weights = (1.0 + 4.0 * ratio / 5.0).astype("float32").values
    flush(f"  Sample weights: min={weights.min():.2f}  "
          f"max={weights.max():.2f}  mean={weights.mean():.2f}")
    return weights


# ─────────────────────────────────────────────────────────────────────────────
# 10. VALIDATION SPLIT
# ─────────────────────────────────────────────────────────────────────────────
def time_split(df: pd.DataFrame, val_frac: float = 0.15):
    cut = df["row_id"].quantile(1 - val_frac)
    tr  = df[df["row_id"] <  cut].copy()
    val = df[df["row_id"] >= cut].copy()
    flush(f"  Train: {len(tr):,}  |  Val: {len(val):,}  "
          f"(cutoff row_id={int(cut):,})")
    return tr, val


# ─────────────────────────────────────────────────────────────────────────────
# 11. MODEL A — LightGBM (delta target + flood weights)
# ─────────────────────────────────────────────────────────────────────────────
LGB_PARAMS = {
    "objective":         "regression",
    "metric":            "rmse",
    "boosting_type":     "gbdt",
    "num_leaves":        511,
    "learning_rate":     0.03,
    "n_estimators":      6000,
    "feature_fraction":  0.75,
    "bagging_fraction":  0.8,
    "bagging_freq":      5,
    "min_child_samples": 20,
    "lambda_l1":         0.05,
    "lambda_l2":         0.5,
    "device_type":       "gpu",
    "max_bin":           63,
    "gpu_platform_id":   0,
    "gpu_device_id":     0,
    "verbose":           -1,
    "seed":              42,
    "n_jobs":            -1,
}


def train_lgb(X_tr, y_tr, X_val, y_val,
              feat_cols, weights=None, tag="LGB") -> tuple:
    flush(f"\n[{tag}] Training on {len(X_tr):,} rows...")
    valid_cats = [c for c in LGB_CAT_COLS if c in feat_cols]
    dtrain = lgb.Dataset(X_tr, label=y_tr, weight=weights,
                         categorical_feature=valid_cats,
                         free_raw_data=False)
    dval   = lgb.Dataset(X_val, label=y_val,
                         categorical_feature=valid_cats,
                         reference=dtrain, free_raw_data=False)
    model = lgb.train(
        LGB_PARAMS, dtrain, valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(stopping_rounds=150, verbose=False),
            lgb.log_evaluation(period=300),
        ],
    )
    best = model.best_iteration
    flush(f"[{tag}] Best iteration: {best}")
    del dtrain, dval; gc.collect()
    return model, best


# ─────────────────────────────────────────────────────────────────────────────
# 12. MODEL B — XGBoost (delta target + flood weights)
# ─────────────────────────────────────────────────────────────────────────────
def _xgb_gpu_params() -> dict:
    major = int(xgb.__version__.split(".")[0])
    return {"device": "cuda", "tree_method": "hist"} if major >= 2 \
        else {"tree_method": "gpu_hist", "gpu_id": 0}


XGB_BASE = {
    "objective":             "reg:squarederror",
    "eval_metric":           "rmse",
    "max_depth":             9,
    "max_leaves":            511,
    "learning_rate":         0.03,
    "n_estimators":          6000,
    "colsample_bytree":      0.75,
    "subsample":             0.8,
    "min_child_weight":      20,
    "reg_alpha":             0.05,
    "reg_lambda":            0.5,
    "max_bin":               64,
    "early_stopping_rounds": 150,
    "seed":                  42,
    "n_jobs":                -1,
    "verbosity":             1,
}


def train_xgb(X_tr, y_tr, X_val, y_val, weights=None) -> tuple:
    flush("\n[XGBoost] Training...")
    model = xgb.XGBRegressor(**{**XGB_BASE, **_xgb_gpu_params()})
    model.fit(X_tr, y_tr,
              sample_weight=weights,
              eval_set=[(X_val, y_val)],
              verbose=300)
    best = getattr(model, "best_iteration",
                   getattr(model, "best_ntree_limit", XGB_BASE["n_estimators"]))
    flush(f"[XGBoost] Best iteration: {best}")
    return model, best


# ─────────────────────────────────────────────────────────────────────────────
# 13. PER-STATION MODELS FOR WORST 10 STATIONS
#     Each bad station gets its own dedicated LGB with higher capacity.
#     Uses the SAME delta target for consistency.
# ─────────────────────────────────────────────────────────────────────────────
LGB_STATION_PARAMS = {
    **LGB_PARAMS,
    "num_leaves":        255,   # smaller — less data per station
    "learning_rate":     0.02,
    "n_estimators":      3000,
    "min_child_samples": 5,     # fewer samples per station
    "feature_fraction":  0.8,
}


def train_station_models(train_df: pd.DataFrame,
                         feat_cols: list,
                         val_frac: float = 0.2) -> dict:
    """
    For each worst station, train a dedicated LGB model on that station only.
    Returns {station_id: lgb_model}
    """
    station_models = {}

    for sid in WORST_STATIONS:
        mask = train_df["station_id"] == sid
        n    = mask.sum()
        if n < 200:
            flush(f"  Station {sid}: only {n} rows — skipping dedicated model")
            continue

        sdf  = train_df[mask].copy()
        cut  = sdf["row_id"].quantile(1 - val_frac)
        s_tr = sdf[sdf["row_id"] <  cut]
        s_val= sdf[sdf["row_id"] >= cut]

        if len(s_tr) < 100 or len(s_val) < 20:
            flush(f"  Station {sid}: insufficient train/val split — skipping")
            continue

        y_tr_s  = make_delta_target(s_tr)
        y_val_s = make_delta_target(s_val)
        w_tr_s  = make_sample_weights(s_tr)

        flush(f"\n  [Station {sid}] n={n}  train={len(s_tr)}  val={len(s_val)}")
        valid_cats = [c for c in LGB_CAT_COLS if c in feat_cols]
        dtrain = lgb.Dataset(s_tr[feat_cols], label=y_tr_s, weight=w_tr_s,
                             categorical_feature=valid_cats,
                             free_raw_data=False)
        dval   = lgb.Dataset(s_val[feat_cols], label=y_val_s,
                             categorical_feature=valid_cats,
                             reference=dtrain, free_raw_data=False)
        m = lgb.train(
            LGB_STATION_PARAMS, dtrain, valid_sets=[dval],
            callbacks=[
                lgb.early_stopping(stopping_rounds=100, verbose=False),
                lgb.log_evaluation(period=500),
            ],
        )

        # Evaluate on val
        delta_pred = m.predict(s_val[feat_cols], num_iteration=m.best_iteration)
        q_pred     = reconstruct_from_delta(
            delta_pred, s_val["streamflow_today_cumecs"].values)
        q_true     = s_val[_TARGET].values.astype("float64")
        flush(f"  [Station {sid}] val KGE: {kge(q_true, q_pred):.6f}")

        station_models[sid] = (m, m.best_iteration)
        del dtrain, dval; gc.collect()

    flush(f"\n  Trained {len(station_models)} per-station models.")
    return station_models


# ─────────────────────────────────────────────────────────────────────────────
# 14. SCIPY BLEND
# ─────────────────────────────────────────────────────────────────────────────
def optimize_blend(preds_dict: dict, y_true: np.ndarray,
                   n_starts: int = 40) -> tuple:
    names = [k for k, v in preds_dict.items() if v is not None]
    P     = np.column_stack([preds_dict[n] for n in names])
    n     = len(names)
    flush(f"\n[Blend] Scipy optimisation over {n} models: {names}")

    def neg_kge(w):
        return -kge(y_true, P @ w)

    np.random.seed(42)
    best_w, best_k = None, -999.0
    for _ in range(n_starts):
        w0  = np.random.dirichlet(np.ones(n))
        res = minimize(neg_kge, w0, method="SLSQP",
                       bounds=[(0, 1)] * n,
                       constraints={"type": "eq",
                                    "fun": lambda w: w.sum() - 1},
                       options={"maxiter": 2000, "ftol": 1e-14})
        if -res.fun > best_k:
            best_k = -res.fun
            best_w = res.x.copy()

    # sanity: equal weights
    w_eq = np.ones(n) / n
    if kge(y_true, P @ w_eq) > best_k:
        best_k = kge(y_true, P @ w_eq)
        best_w = w_eq

    best_w = np.clip(best_w, 0, 1); best_w /= best_w.sum()
    result = {name: float(w) for name, w in zip(names, best_w)}
    flush(f"  Weights: {result}  KGE: {best_k:.6f}")
    return result, best_k


def blend_predictions(preds: dict, weights: dict) -> np.ndarray:
    out = np.zeros(
        len(next(v for v in preds.values() if v is not None)), dtype="float64")
    for name, w in weights.items():
        if preds.get(name) is not None:
            out += w * preds[name].astype("float64")
    return out


# ─────────────────────────────────────────────────────────────────────────────
# 15. PER-STATION BIAS CORRECTION
# ─────────────────────────────────────────────────────────────────────────────
def fit_station_bias(val_df: pd.DataFrame,
                     val_pred: np.ndarray) -> pd.Series:
    tmp = val_df[["station_id", _TARGET]].copy()
    tmp["pred"] = val_pred
    scales = {}
    for sid, g in tmp.groupby("station_id"):
        raw = g[_TARGET].mean() / (g["pred"].mean() + 1e-10)
        clamp = (0.2, 5.0) if sid in WORST_STATIONS else (0.5, 2.0)
        scales[sid] = float(np.clip(raw, *clamp))
    s = pd.Series(scales)
    flush(f"  Bias scales: min={s.min():.4f} max={s.max():.4f} mean={s.mean():.4f}")
    return s


def apply_station_bias(df: pd.DataFrame, pred: np.ndarray,
                       scales: pd.Series) -> np.ndarray:
    out = pred.copy().astype("float64")
    for sid, scale in scales.items():
        out[df["station_id"].values == sid] *= scale
    return out


# ─────────────────────────────────────────────────────────────────────────────
# 16. FEATURE IMPORTANCE
# ─────────────────────────────────────────────────────────────────────────────
def show_importance(model, top_n=25):
    imp = pd.DataFrame({
        "feature": model.feature_name(),
        "gain":    model.feature_importance("gain"),
    }).sort_values("gain", ascending=False).head(top_n)
    flush(f"\n{'Feature':50s}  Gain")
    flush("-" * 70)
    mg = imp["gain"].iloc[0]
    for _, row in imp.iterrows():
        bar = "█" * int(row["gain"] / mg * 35)
        flush(f"  {row['feature']:48s}  {bar}")


# ─────────────────────────────────────────────────────────────────────────────

# ─────────────────────────────────────────────────────────────────────────────
# 17. DIAGNOSTICS
# ─────────────────────────────────────────────────────────────────────────────
def run_diagnostics(val_df, y_val_raw, y_val_pers,
                    blend_val_corrected, lgb_val_q, xgb_val_q,
                    lgb_model, diag_dir):
    import warnings; warnings.filterwarnings("ignore")
    os.makedirs(diag_dir, exist_ok=True)
    flush(f"\n[DIAG] Saving diagnostic plots → {diag_dir}")

    MONTH_LABELS = ["Jan","Feb","Mar","Apr","May","Jun",
                    "Jul","Aug","Sep","Oct","Nov","Dec"]

    errors     = y_val_raw - blend_val_corrected
    rel_errors = np.abs(errors) / (y_val_raw + 1)
    vdf = val_df.copy()
    vdf["error"] = np.abs(errors); vdf["rel_error"] = rel_errors
    vdf["pred"] = blend_val_corrected; vdf["signed_err"] = errors

    station_kge = {}
    for sid in vdf["station_id"].unique():
        m = vdf["station_id"].values == sid
        if m.sum() >= 20:
            station_kge[sid] = kge(y_val_raw[m], blend_val_corrected[m])
    kge_series = pd.Series(station_kge)

    # ── Page 1: Overview
    fig = plt.figure(figsize=(16,12))
    fig.suptitle("v9 Diagnostic — Overview", fontsize=15, fontweight="bold")
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)

    ax = fig.add_subplot(gs[0,0])
    sc = ax.scatter(y_val_raw, errors, alpha=0.04, s=1,
                    c=np.log1p(y_val_raw), cmap="plasma")
    ax.axhline(0, color="red", lw=1)
    ax.set_xscale("symlog"); ax.set_xlabel("True flow (cumecs)")
    ax.set_ylabel("Error  (true − pred)")
    ax.set_title("Error vs Flow Magnitude\n(colour = log flow)")
    plt.colorbar(sc, ax=ax, label="log1p(Q)")

    ax = fig.add_subplot(gs[0,1])
    lim = max(float(y_val_raw.max()), float(blend_val_corrected.max()))*1.05
    ax.scatter(y_val_raw, blend_val_corrected, alpha=0.04, s=1, color="darkorange")
    ax.plot([0,lim],[0,lim],"r--",lw=1,label="Perfect")
    ax.set_xscale("symlog"); ax.set_yscale("symlog")
    ax.set_xlabel("True flow (cumecs)"); ax.set_ylabel("Predicted (cumecs)")
    ax.set_title("Predicted vs Actual  (log–log)"); ax.legend(fontsize=8)

    ax = fig.add_subplot(gs[1,0])
    monthly = vdf.groupby("month")["rel_error"].mean()
    ax.bar(monthly.index, monthly.values,
           color=["#e74c3c" if v > monthly.mean()*1.3 else "#2ecc71"
                  for v in monthly.values])
    ax.axhline(monthly.mean(), color="navy", lw=1, ls="--",
               label=f"Mean={monthly.mean():.4f}")
    ax.set_xticks(range(1,13)); ax.set_xticklabels(MONTH_LABELS, fontsize=8)
    ax.set_ylabel("Mean relative error")
    ax.set_title("Relative Error by Month\n(red = significantly worse)")
    ax.legend(fontsize=8)

    ax = fig.add_subplot(gs[1,1])
    ax.hist(kge_series.values, bins=40, color="mediumpurple", edgecolor="white")
    ax.axvline(kge_series.median(), color="red", ls="--",
               label=f"Median: {kge_series.median():.4f}")
    ax.axvline(kge_series.quantile(0.05), color="orange", ls=":",
               label=f"P5: {kge_series.quantile(0.05):.4f}")
    ax.set_xlabel("Per-station KGE"); ax.set_ylabel("Count")
    ax.set_title("Per-Station KGE Distribution"); ax.legend(fontsize=8)

    fig.savefig(os.path.join(diag_dir, "page1_overview.png"), dpi=150, bbox_inches="tight")
    plt.close(fig); flush("  ✓ page1_overview.png")

    # ── Page 2: KGE Components
    fig = plt.figure(figsize=(16,12))
    fig.suptitle("v9 Diagnostic — KGE Components", fontsize=15, fontweight="bold")
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)

    try:
        vdf["flow_decile"] = pd.qcut(y_val_raw, q=10, labels=False, duplicates="drop")
    except Exception:
        vdf["flow_decile"] = pd.cut(y_val_raw, bins=10, labels=False)

    decile_stats = []
    for d, g in vdf.groupby("flow_decile"):
        yo = g[_TARGET].values.astype("float64")
        yp = g["pred"].values.astype("float64")
        c = kge_components(yo, yp); c["decile"] = d; c["mean_flow"] = yo.mean()
        decile_stats.append(c)
    ds = pd.DataFrame(decile_stats).set_index("decile")

    ax = fig.add_subplot(gs[0,0])
    ax.bar(ds.index, ds["r"],
           color=["#e74c3c" if v<0.999 else "#2ecc71" for v in ds["r"]])
    ax.axhline(1.0, color="black", lw=1, ls="--")
    ax.set_ylim(max(0.98, ds["r"].min()-0.002), 1.002)
    ax.set_xlabel("Flow decile  (0=low, 9=peak)"); ax.set_ylabel("r")
    ax.set_title("KGE r by Flow Decile")

    ax = fig.add_subplot(gs[0,1])
    ax.bar(ds.index, ds["alpha"],
           color=["#e74c3c" if abs(v-1)>0.01 else "#2ecc71" for v in ds["alpha"]])
    ax.axhline(1.0, color="black", lw=1, ls="--")
    ax.set_xlabel("Flow decile"); ax.set_ylabel("alpha")
    ax.set_title("KGE alpha by Flow Decile\n(>1 = over-predict variance)")

    ax = fig.add_subplot(gs[1,0])
    ax.bar(ds.index, ds["beta"],
           color=["#e74c3c" if abs(v-1)>0.01 else "#2ecc71" for v in ds["beta"]])
    ax.axhline(1.0, color="black", lw=1, ls="--")
    ax.set_xlabel("Flow decile"); ax.set_ylabel("beta")
    ax.set_title("KGE beta by Flow Decile\n(>1 = over-predict mean)")

    ax = fig.add_subplot(gs[1,1])
    lgb_kge_list, xgb_kge_list, blend_kge_list = [], [], []
    for sid in vdf["station_id"].unique():
        m = vdf["station_id"].values == sid
        if m.sum() < 20: continue
        yo = y_val_raw[m]
        lgb_kge_list.append(kge(yo, lgb_val_q[m]))
        xgb_kge_list.append(kge(yo, xgb_val_q[m]))
        blend_kge_list.append(kge(yo, blend_val_corrected[m]))
    ax.scatter(lgb_kge_list,  blend_kge_list, alpha=0.4, s=8, color="steelblue",  label="LGB vs Blend")
    ax.scatter(xgb_kge_list,  blend_kge_list, alpha=0.4, s=8, color="darkorange", label="XGB vs Blend")
    lims = [min(min(lgb_kge_list), min(xgb_kge_list))-0.001, 1.001]
    ax.plot(lims, lims, "r--", lw=1)
    ax.set_xlabel("Individual model KGE"); ax.set_ylabel("Blend KGE")
    ax.set_title("Per-Station: Individual vs Blend"); ax.legend(fontsize=8)

    fig.savefig(os.path.join(diag_dir, "page2_kge_components.png"), dpi=150, bbox_inches="tight")
    plt.close(fig); flush("  ✓ page2_kge_components.png")

    # ── Page 3: Flood Regime
    fig = plt.figure(figsize=(16,12))
    fig.suptitle("v9 Diagnostic — Flood Regime Analysis", fontsize=15, fontweight="bold")
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.35)

    ax = fig.add_subplot(gs[0,0])
    for flag, label, color in [("is_rising","Rising","#e74c3c"),("is_receding","Receding","#3498db")]:
        if flag in vdf.columns:
            m = vdf[flag].values == 1
            ax.hist(errors[m], bins=60, alpha=0.5, color=color,
                    label=f"{label} (n={m.sum():,})", density=True)
    if "is_rising" in vdf.columns and "is_receding" in vdf.columns:
        ms = (vdf["is_rising"].values==0)&(vdf["is_receding"].values==0)
        ax.hist(errors[ms], bins=60, alpha=0.4, color="gray",
                label=f"Stable (n={ms.sum():,})", density=True)
    ax.axvline(0, color="black", lw=1)
    ax.set_xlim(-500, 500); ax.set_xlabel("Error"); ax.set_ylabel("Density")
    ax.set_title("Error by Flow Regime"); ax.legend(fontsize=7)

    ax = fig.add_subplot(gs[0,1])
    regime_kge = {}
    if "is_rising" in vdf.columns:
        for flag, label in [("is_rising","Rising"),("is_receding","Receding"),("is_flooding","Flooding")]:
            m = vdf[flag].values == 1
            if m.sum() > 50:
                regime_kge[label] = kge(y_val_raw[m], blend_val_corrected[m])
        ms = ((vdf["is_rising"].values==0)&(vdf["is_receding"].values==0)&(vdf["is_flooding"].values==0))
        if ms.sum() > 50:
            regime_kge["Stable"] = kge(y_val_raw[ms], blend_val_corrected[ms])
    cmap_r = {"Rising":"#e74c3c","Receding":"#3498db","Flooding":"#e67e22","Stable":"#2ecc71"}
    if regime_kge:
        ax.bar(regime_kge.keys(), regime_kge.values(),
               color=[cmap_r.get(k,"gray") for k in regime_kge])
        ax.axhline(1.0, color="black", lw=1, ls="--")
        ax.set_ylim(min(regime_kge.values())-0.002, 1.001)
        for i,(k,v) in enumerate(regime_kge.items()):
            ax.text(i, v+0.0001, f"{v:.4f}", ha="center", fontsize=8)
    ax.set_ylabel("KGE"); ax.set_title("KGE by Flow Regime")

    ax = fig.add_subplot(gs[0,2])
    w_sids = [s for s in WORST_STATIONS if (vdf["station_id"].values==s).sum()>10]
    w_kge  = [kge(y_val_raw[vdf["station_id"].values==s],
                  blend_val_corrected[vdf["station_id"].values==s]) for s in w_sids]
    cw = ["#e74c3c" if k<0.995 else "#f39c12" if k<0.998 else "#2ecc71" for k in w_kge]
    ax.barh([str(s) for s in w_sids], w_kge, color=cw)
    ax.axvline(0.999, color="navy", lw=1, ls="--", label="0.999 target")
    ax.set_xlabel("KGE"); ax.set_title("Worst Station KGE"); ax.legend(fontsize=8)
    for i,(s,k) in enumerate(zip(w_sids,w_kge)):
        ax.text(k+0.0001, i, f"{k:.4f}", va="center", fontsize=7)

    ax = fig.add_subplot(gs[1,0])
    if "flow_lag1_ratio" in vdf.columns:
        ratio = vdf["flow_lag1_ratio"].values.clip(0,5)
        ax.scatter(ratio, errors, alpha=0.03, s=1, color="purple")
        ax.axhline(0, color="red", lw=1)
        ax.set_xlabel("flow_lag1_ratio  (Q_today / Q_yesterday)")
        ax.set_ylabel("Error"); ax.set_title("Error vs Rising Limb Signal")

    ax = fig.add_subplot(gs[1,1])
    weekly = vdf.groupby("week_of_year")["rel_error"].mean()
    ax.plot(weekly.index, weekly.values, color="teal", lw=1.5)
    ax.fill_between(weekly.index, weekly.values, alpha=0.3, color="teal")
    ax.axvspan(21, 38, alpha=0.1, color="red", label="Peak monsoon")
    ax.set_xlabel("Week of year"); ax.set_ylabel("Mean relative error")
    ax.set_title("Relative Error by Week"); ax.legend(fontsize=8)

    ax = fig.add_subplot(gs[1,2])
    ax.scatter(vdf["soil_saturation_score"].values, rel_errors,
               alpha=0.03, s=1, color="brown")
    ax.set_xlabel("Soil saturation score"); ax.set_ylabel("Relative error")
    ax.set_title("Relative Error vs Soil Saturation")

    fig.savefig(os.path.join(diag_dir, "page3_flood_regime.png"), dpi=150, bbox_inches="tight")
    plt.close(fig); flush("  ✓ page3_flood_regime.png")

    # ── Page 4: Feature Importance
    fig, ax = plt.subplots(figsize=(12,14))
    fig.suptitle("v9 — LightGBM Feature Importance (Gain)", fontsize=14, fontweight="bold")
    imp = pd.DataFrame({
        "feature": lgb_model.feature_name(),
        "gain":    lgb_model.feature_importance("gain"),
    }).sort_values("gain", ascending=False).head(40)
    cmap_feat = []
    for f in imp["feature"]:
        if any(x in f for x in ["lag","roll","_d1","_d2","ewm","ratio"]):
            cmap_feat.append("#e74c3c")
        elif "rain" in f: cmap_feat.append("#3498db")
        elif "station" in f: cmap_feat.append("#9b59b6")
        elif any(x in f for x in ["sat","soil"]): cmap_feat.append("#27ae60")
        else: cmap_feat.append("#95a5a6")
    ax.barh(imp["feature"][::-1], imp["gain"][::-1], color=cmap_feat[::-1])
    ax.set_xlabel("Gain")
    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(facecolor="#e74c3c", label="Lag/temporal"),
        Patch(facecolor="#3498db", label="Rain"),
        Patch(facecolor="#9b59b6", label="Station"),
        Patch(facecolor="#27ae60", label="Soil/saturation"),
        Patch(facecolor="#95a5a6", label="Other"),
    ], loc="lower right", fontsize=9)
    fig.tight_layout()
    fig.savefig(os.path.join(diag_dir, "page4_feature_importance.png"), dpi=150, bbox_inches="tight")
    plt.close(fig); flush("  ✓ page4_feature_importance.png")

    # ── Text summary
    flush("\n" + "="*60)
    flush("DIAGNOSTIC SUMMARY")
    flush("="*60)
    flush(f"  Overall KGE   : {kge(y_val_raw, blend_val_corrected):.6f}")
    flush(f"  {kge_components(y_val_raw, blend_val_corrected)}")
    flush(f"  Station KGE   median={kge_series.median():.4f}  "
          f"min={kge_series.min():.4f}  p5={kge_series.quantile(0.05):.4f}")
    comps_all = kge_components(y_val_raw, blend_val_corrected)
    gaps = {k: abs(1.0-v) for k,v in comps_all.items() if k != "KGE"}
    worst_comp = max(gaps, key=gaps.get)
    advice = {
        "r":     "Add more lag features or lower LR",
        "alpha": "Variance mismatch — model smoothing flood peaks",
        "beta":  "Bias — station correction not fully working",
    }
    flush(f"  ► Bottleneck: {worst_comp} (gap={gaps[worst_comp]:.6f}) → {advice[worst_comp]}")
    flush("="*60)
    flush(f"  Plots saved → {diag_dir}")
    return kge_series

# 17. MAIN PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
def main():
    flush("=" * 70)
    flush("  FLOOD STREAMFLOW PREDICTION  v9")
    flush("  Delta target + flood weights + per-station models")
    flush("=" * 70)

    # ── Load
    train = load_data(TRAIN_PATH, is_train=True)
    test  = load_data(TEST_PATH,  is_train=False)

    # ── Station
    flush("\n[STATION] Identifying gauge stations...")
    train, enc = add_station_id(train)
    test,  _   = add_station_id(test, enc=enc)
    flush(f"  Unique stations: {train['station_id'].nunique()}")
    train, test = add_station_stats(train, test)

    # ── Lags
    flush("\n[LAGS] Adding lag features...")
    train = add_lag_features(train)
    test  = add_lag_features(test)

    # ── Features
    flush("\n[FEATURES] Engineering...")
    train = engineer_features(train)
    test  = engineer_features(test)
    feat_cols = get_feature_cols(train)
    flush(f"  Total features: {len(feat_cols)}")

    # ── Delta target (THE KEY CHANGE)
    train["delta_target"] = make_delta_target(train)
    flush(f"\n[TARGET] Delta target stats:")
    flush(f"  mean={train['delta_target'].mean():.4f}  "
          f"std={train['delta_target'].std():.4f}  "
          f"min={train['delta_target'].min():.4f}  "
          f"max={train['delta_target'].max():.4f}")
    flush(f"  (compare to log1p(Q) std ≈ large — delta std is much smaller)")

    # ── Persistence baseline
    flush("\n[BASELINE] Persistence:")
    tr_tmp, val_tmp = time_split(train)
    y_pers = val_tmp["streamflow_today_cumecs"].values.astype("float64")
    y_true = val_tmp[_TARGET].values.astype("float64")
    for k, v in kge_components(y_true, y_pers).items():
        flush(f"  {k}: {v}")
    del tr_tmp, val_tmp; gc.collect()

    # ── Val split
    flush("\n[SPLIT] Time-based validation split...")
    tr, val = time_split(train)

    X_tr_df   = tr[feat_cols]
    y_tr_delta = tr["delta_target"].values.astype("float32")
    w_tr       = make_sample_weights(tr)

    X_val_df   = val[feat_cols]
    y_val_raw  = val[_TARGET].values.astype("float64")
    y_val_pers = val["streamflow_today_cumecs"].values.astype("float64")
    q_today_val = val["streamflow_today_cumecs"].values.astype("float64")

    # Delta target for val (for model training reference only)
    y_val_delta = make_delta_target(val)

    flush("\nPersistence on val:")
    for k, v in kge_components(y_val_raw, y_val_pers).items():
        flush(f"  {k}: {v}")

    # ── MODEL A: LightGBM
    lgb_model, lgb_best = train_lgb(
        X_tr_df, y_tr_delta, X_val_df, y_val_delta,
        feat_cols, weights=w_tr, tag="LGB-Global")
    lgb_val_delta = lgb_model.predict(X_val_df, num_iteration=lgb_best)
    lgb_val_q     = reconstruct_from_delta(lgb_val_delta, q_today_val)
    flush(f"LGB val KGE: {kge(y_val_raw, lgb_val_q):.6f}  "
          f"{kge_components(y_val_raw, lgb_val_q)}")
    show_importance(lgb_model)
    gc.collect()

    # ── MODEL B: XGBoost
    xgb_model, xgb_best = train_xgb(
        X_tr_df, y_tr_delta, X_val_df, y_val_delta, weights=w_tr)
    xgb_val_delta = xgb_model.predict(X_val_df)
    xgb_val_q     = reconstruct_from_delta(xgb_val_delta, q_today_val)
    flush(f"XGB val KGE: {kge(y_val_raw, xgb_val_q):.6f}  "
          f"{kge_components(y_val_raw, xgb_val_q)}")
    gc.collect()

    # ── BLEND of global models
    val_preds_q  = {"lgb": lgb_val_q, "xgb": xgb_val_q}
    best_weights, _ = optimize_blend(val_preds_q, y_val_raw)
    blend_val_q  = blend_predictions(val_preds_q, best_weights)
    flush(f"\nGlobal blend KGE: {kge(y_val_raw, blend_val_q):.6f}")

    # ── PER-STATION MODELS for worst stations
    flush("\n[STATION MODELS] Training dedicated models for worst stations...")
    station_models = train_station_models(tr, feat_cols)

    # Apply station model overrides to val predictions
    blend_val_final = blend_val_q.copy()
    for sid, (sm, sm_best) in station_models.items():
        mask = val["station_id"].values == sid
        if not mask.any():
            continue
        delta_s = sm.predict(X_val_df[mask], num_iteration=sm_best)
        q_s     = reconstruct_from_delta(delta_s, q_today_val[mask])
        # Blend 50/50 global + station-specific (don't fully trust station model)
        blend_val_final[mask] = 0.5 * blend_val_q[mask] + 0.5 * q_s
        k_s = kge(y_val_raw[mask], blend_val_final[mask])
        flush(f"  Station {sid:4d} after station model: KGE={k_s:.6f}")

    flush(f"\nAfter station models KGE: {kge(y_val_raw, blend_val_final):.6f}")

    # ── Per-station bias correction
    flush("\n[BIAS] Per-station correction...")
    station_scales      = fit_station_bias(val, blend_val_final)
    blend_val_corrected = apply_station_bias(val, blend_val_final, station_scales)

    # ── Final diagnostics
    flush("\n=== FINAL VAL DIAGNOSTICS ===")
    comps = kge_components(y_val_raw, blend_val_corrected)
    for k, v in comps.items():
        flush(f"  {k}: {v}")
    flush(f"  NSE: {nse(y_val_raw, blend_val_corrected):.6f}")
    flush(f"  Δ vs persistence  : "
          f"{kge(y_val_raw, blend_val_corrected) - kge(y_val_raw, y_val_pers):+.6f}")
    flush(f"  Δ vs v8 (0.99726) : "
          f"{kge(y_val_raw, blend_val_corrected) - 0.99726:+.6f}")

    flush("\nWorst station check:")
    for sid in sorted(WORST_STATIONS):
        mask = val["station_id"].values == sid
        if mask.sum() > 10:
            k = kge(y_val_raw[mask], blend_val_corrected[mask])
            flush(f"  Station {sid:4d}: KGE={k:.6f}  n={mask.sum()}")


    # ── DIAGNOSTICS  (runs after val is fully evaluated, before retrain)
    flush("\n[DIAG] Running post-val diagnostics...")
    run_diagnostics(
        val_df              = val,
        y_val_raw           = y_val_raw,
        y_val_pers          = y_val_pers,
        blend_val_corrected = blend_val_corrected,
        lgb_val_q           = lgb_val_q,
        xgb_val_q           = xgb_val_q,
        lgb_model           = lgb_model,
        diag_dir            = DIAG_DIR,
    )

    # ── RETRAIN ON FULL DATA
    flush("\n[RETRAIN] Full training set...")
    full_delta  = train["delta_target"].values.astype("float32")
    full_w      = make_sample_weights(train)
    valid_cats  = [c for c in LGB_CAT_COLS if c in feat_cols]

    # LGB full
    dfull = lgb.Dataset(train[feat_cols], label=full_delta, weight=full_w,
                        categorical_feature=valid_cats, free_raw_data=False)
    lp = {**LGB_PARAMS, "n_estimators": int(lgb_best * 1.10)}
    lp.pop("metric", None)
    lgb_final = lgb.train(lp, dfull, callbacks=[lgb.log_evaluation(300)])
    del dfull; gc.collect()

    # XGB full
    xgb_params_final = {**XGB_BASE, **_xgb_gpu_params(),
                        "n_estimators": int(xgb_best * 1.10)}
    xgb_params_final.pop("early_stopping_rounds", None)
    xgb_final = xgb.XGBRegressor(**xgb_params_final)
    xgb_final.fit(train[feat_cols], full_delta,
                  sample_weight=full_w, verbose=300)
    gc.collect()

    # Station models full retrain
    station_models_full = {}
    for sid in WORST_STATIONS:
        mask = train["station_id"] == sid
        if mask.sum() < 200:
            continue
        sdf    = train[mask]
        y_s    = sdf["delta_target"].values.astype("float32")
        w_s    = make_sample_weights(sdf)
        valid_cats_s = [c for c in LGB_CAT_COLS if c in feat_cols]
        # Find best iter from val training
        best_s = station_models.get(sid, (None, 1000))[1]
        dfull_s = lgb.Dataset(sdf[feat_cols], label=y_s, weight=w_s,
                              categorical_feature=valid_cats_s,
                              free_raw_data=False)
        lp_s = {**LGB_STATION_PARAMS, "n_estimators": int(best_s * 1.10)}
        lp_s.pop("metric", None)
        m_s = lgb.train(lp_s, dfull_s, callbacks=[lgb.log_evaluation(500)])
        station_models_full[sid] = m_s
        del dfull_s; gc.collect()

    # ── TEST PREDICTIONS
    flush("\n[PREDICT] Test predictions...")
    X_test        = test[feat_cols]
    q_today_test  = test["streamflow_today_cumecs"].values.astype("float64")

    lgb_test_delta = lgb_final.predict(X_test)
    xgb_test_delta = xgb_final.predict(X_test)

    lgb_test_q = reconstruct_from_delta(lgb_test_delta, q_today_test)
    xgb_test_q = reconstruct_from_delta(xgb_test_delta, q_today_test)

    test_preds   = {"lgb": lgb_test_q, "xgb": xgb_test_q}
    test_blend_q = blend_predictions(test_preds, best_weights)

    # Apply station model overrides to test
    test_final = test_blend_q.copy()
    for sid, sm in station_models_full.items():
        mask = test["station_id"].values == sid
        if not mask.any():
            continue
        delta_s = sm.predict(X_test[mask])
        q_s     = reconstruct_from_delta(delta_s, q_today_test[mask])
        test_final[mask] = 0.5 * test_blend_q[mask] + 0.5 * q_s

    test_final = apply_station_bias(test, test_final, station_scales)

    flush(f"  Test range: [{test_final.min():.2f}, {test_final.max():.2f}]")
    flush(f"  Test mean : {test_final.mean():.2f}")

    # ── Submission
    sub = pd.DataFrame({
        "row_id":                     test["row_id"].values,
        "streamflow_tomorrow_cumecs": test_final,
    })
    sub.to_csv(SUB_PATH, index=False)
    flush(f"\n[DONE] Saved {len(sub):,} rows → {SUB_PATH}")
    flush(sub.head().to_string())

    lgb_final.save_model(os.path.join(SUB_DIR, "lgb_model.txt"))
    xgb_final.save_model(os.path.join(SUB_DIR, "xgb_model.json"))
    for sid, m in station_models_full.items():
        m.save_model(os.path.join(SUB_DIR, f"lgb_station_{sid}.txt"))

    flush("\n" + "=" * 70)
    flush("  PIPELINE COMPLETE")
    flush(f"  Val KGE : {kge(y_val_raw, blend_val_corrected):.6f}")
    flush(f"  Weights : {best_weights}")
    flush("=" * 70)
    return sub


if __name__ == "__main__":
    main()

  FLOOD STREAMFLOW PREDICTION  v9
  Delta target + flood weights + per-station models

Loading train...
  2,571,055 rows × 34 cols | 329 MB

Loading test...
  642,764 rows × 33 cols | 80 MB

[STATION] Identifying gauge stations...
  Unique stations: 367
  station_mean_flow range: [88, 5147]

[LAGS] Adding lag features...
  Adding lag features...
  Shape after lags: (2571055, 53)
  Adding lag features...
  Shape after lags: (642764, 52)

[FEATURES] Engineering...
  Total features: 94

[TARGET] Delta target stats:
  mean=0.0000  std=0.0628  min=-0.7925  max=1.6696
  (compare to log1p(Q) std ≈ large — delta std is much smaller)

[BASELINE] Persistence:
  Train: 2,185,396  |  Val: 385,659  (cutoff row_id=2,185,395)
  KGE: 0.997882
  r: 0.9979
  alpha: 1.0
  beta: 0.9999

[SPLIT] Time-based validation split...
  Train: 2,185,396  |  Val: 385,659  (cutoff row_id=2,185,395)
  Sample weights: min=1.08  max=5.00  mean=1.79

Persistence on val:
  KGE: 0.997882
  r: 0.9979
  alpha: 1.0
  beta: 0.